# Trend Projection to 2030

This notebook projects adaptation aid forward to 2030 to illustrate where current allocation
trajectories lead. Two views are produced: a per-country linear trend, fitted on each country's
2010 to 2023 aid series and extrapolated to 2030, which feeds the application's country projection
panel; and a regional aggregate trend, which is more stable because it averages over country-level
noise and pairs with the Paris Agreement finding on regional allocation. All projections are naive
linear extrapolations of historical trend, presented as illustrative scenarios under a
constant-trend assumption rather than as forecasts.

## Step 1 — Load the analytical inputs

The projection is a descriptive exercise requiring only an aid value and a year, so it is not bound
by the model's listwise deletion. The aid trajectory is therefore taken from the full pre-model
panel rather than the complete-case modelling sample. This recovers countries whose aid record is
complete but whose covariates were missing, most importantly Eritrea, the most underfunded
low-income country, which the model retained for only two years but which has a full fourteen-year
aid history. Country-level attributes from Notebook 03 (region, allocation profile, income tier) are
attached. Aid is modelled on the log scale, using the log of one plus aid to accommodate zero-aid
years, and exponentiated back to dollars for display, which keeps projections positive. Countries
with fewer than five years of aid history cannot support a linear trend and are excluded; four high
income countries that received adaptation aid in only a single year fall into this category and are
reported as insufficient-data cases. The model thus uses the complete-case sample while the
projection uses the full aid record: two samples for two distinct purposes.

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

root = Path.cwd()
while root.parent != root and not (root / ".venv").exists():
    root = root.parent

cinfo = pd.read_csv(root / "data" / "processed" / "country_scored.csv")
panel_full = pd.read_csv(root / "data" / "processed" / "panel.csv")

AIDCOL = "adaptation_aid_usd_m"
ids = set(cinfo["iso3"])

# full aid record for the 138 analysis countries (projection needs only aid + year)
aidp = (panel_full.loc[panel_full["iso3"].isin(ids), ["iso3", "year", AIDCOL]]
                  .dropna(subset=[AIDCOL]).copy())
aidp["log_aid"] = np.log1p(aidp[AIDCOL])
aidp = aidp.merge(cinfo[["iso3", "country", "region", "profile", "income_tier"]],
                  on="iso3", how="left")

MIN_YRS = 5
pts = aidp.groupby("iso3")["year"].nunique()
projectable = pts[pts >= MIN_YRS].index
excluded    = pts[pts < MIN_YRS]

print(f"Aid records: {len(aidp):,} rows, {aidp['iso3'].nunique()} countries")
print(f"Projectable (>= {MIN_YRS} yrs): {len(projectable)} | excluded: {len(excluded)}")
print("\nExcluded (insufficient aid history — reported, not projected):")
print(excluded.to_frame("n_years").join(cinfo.set_index("iso3")[["country", "income_tier"]]).to_string())

Aid records: 1,827 rows, 138 countries
Projectable (>= 5 yrs): 134 | excluded: 4

Excluded (insufficient aid history — reported, not projected):
      n_years              country income_tier
iso3                                          
BRB         1             Barbados           H
HRV         1              Croatia           H
OMN         1                 Oman           H
TTO         1  Trinidad and Tobago           H


## Step 2 — Per-country trend fit and 2030 projection

For each of the 134 projectable countries, a linear trend is fitted to its log-aid history and
extrapolated to 2030, then exponentiated to dollars. The fitted 2023 value and projected 2030 value
define the trajectory, summarised as an implied annual growth rate and a 2023 to 2030 percentage
change. Because adaptation aid is volatile and each series spans at most fourteen points, the
statistical significance of each trend is recorded; a country whose trend is not significant is
treated as having no reliable trajectory rather than a genuine rise or fall. The slope and intercept
are retained so the application can reconstruct the full historical and projected line. The
distribution of trajectory directions across allocation profiles is then examined, a forward-looking
question: whether the countries currently identified as underfunded are on rising aid paths that
would narrow the gap, or on flat or falling paths that would entrench it.

In [4]:
from scipy import stats

def fit_country(g):
    r = stats.linregress(g["year"], g["log_aid"])
    return pd.Series({"n_years": g["year"].nunique(), "slope": r.slope,
                      "intercept": r.intercept, "trend_p": r.pvalue, "trend_r2": r.rvalue**2})

projectable_df = aidp[aidp["iso3"].isin(projectable)]
proj = (projectable_df
        .groupby(["iso3", "country", "region", "profile", "income_tier"], as_index=False)
        .apply(fit_country, include_groups=False))

at = lambda year: np.expm1(proj["intercept"] + proj["slope"] * year)
proj["aid_2023_fit_usd_m"]  = at(2023)
proj["aid_2030_proj_usd_m"] = at(2030)
proj["annual_pct"]       = (np.exp(proj["slope"]) - 1) * 100
proj["pct_change_23_30"] = (proj["aid_2030_proj_usd_m"] / proj["aid_2023_fit_usd_m"] - 1) * 100
proj["trend_significant"] = proj["trend_p"] < 0.05
proj["direction"] = np.select([proj["annual_pct"] > 1, proj["annual_pct"] < -1],
                              ["Rising", "Falling"], default="Flat")

OUT = root / "data" / "processed" / "country_projection.csv"
proj.to_csv(OUT, index=False)
print("saved:", OUT, "| countries:", len(proj))

print("\nTrajectory direction:")
print(proj["direction"].value_counts().to_string())
print(f"Statistically significant trends: {int(proj['trend_significant'].sum())} of {len(proj)}")

print("\nDirection × allocation profile (forward-looking: are underfunded countries on rising paths?):")
print(pd.crosstab(proj["profile"], proj["direction"])
        .reindex(["Chronically Underfunded","Underfunded","Adequately Funded","Over-Resourced"]).to_string())

saved: /Users/sanjogkadayat/climate_adaptation_capstone/data/processed/country_projection.csv | countries: 134

Trajectory direction:
direction
Rising     116
Falling     13
Flat         5
Statistically significant trends: 71 of 134

Direction × allocation profile (forward-looking: are underfunded countries on rising paths?):
direction                Falling  Flat  Rising
profile                                       
Chronically Underfunded        4     3      27
Underfunded                    6     1      24
Adequately Funded              2     0      32
Over-Resourced                 1     1      33
